# Solution to p17

This notebook reconstructs the seeded baseline directly from the learner fixture. The split wrappers accept only the complete stored-order ID tuple. `compute_loss` converts the actual tensors at its seam back to ordered NumPy fields and asks the fixture to resolve feature fingerprints and aligned target fingerprints on every call; it therefore does not trust declared IDs. The class token is a shared expanded parameter, all five positioned tokens participate in self-attention, and only the normalized residual class token reaches the head.

In [ ]:
import importlib.util
import random
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260901
ATOL = 1e-5
RTOL = 1e-5
TRAIN_IDS = ('vit-train-00','vit-train-01','vit-train-02','vit-train-03')
HELDOUT_IDS = ('vit-heldout-00','vit-heldout-01')

fixture_path = Path("units/B2-021-cross-modal-transformers-vision/data/vision_fixture.py")
fixture_spec = importlib.util.spec_from_file_location("p17_vision_fixture", fixture_path)
vision_fixture = importlib.util.module_from_spec(fixture_spec)
sys.modules[fixture_spec.name] = vision_fixture
fixture_spec.loader.exec_module(vision_fixture)

def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)

def _to_torch(fields):
    return {name: torch.from_numpy(value.copy()) for name, value in fields.items()}

def _to_numpy(fields):
    converted = {}
    for name, value in fields.items():
        if not isinstance(value, torch.Tensor):
            raise TypeError("batch fields must be Torch tensors")
        converted[name] = value.detach().cpu().contiguous().numpy().copy()
    return converted

def build_train_batch(ids):
    if tuple(ids) != TRAIN_IDS:
        raise ValueError("train IDs must equal the immutable stored-order split")
    features, targets = vision_fixture.build_task_batch('vit', tuple(ids))
    return _to_torch(features), _to_torch(targets)

def build_heldout_batch(ids):
    if tuple(ids) != HELDOUT_IDS:
        raise ValueError("held-out IDs must equal the immutable stored-order split")
    features, targets = vision_fixture.build_task_batch('vit', tuple(ids))
    return _to_torch(features), _to_torch(targets)

def validate_actual(features, targets):
    role, ids = vision_fixture.validate_actual_batch('vit', _to_numpy(features), _to_numpy(targets))
    images, labels = features.get("image"), targets.get("label")
    if images is None or labels is None or images.ndim != 4 or images.shape[1:] != (1,4,4):
        raise ValueError("images must be (B,1,4,4)")
    if labels.shape != (images.shape[0],):
        raise ValueError("labels must be (B,)")
    if images.dtype != torch.float32 or labels.dtype != torch.int64:
        raise ValueError("image/label dtype mismatch")
    if images.device.type != "cpu" or labels.device.type != "cpu":
        raise ValueError("batch must be on CPU")
    return role, ids

def patchify(images):
    B, C, H, W = images.shape
    separated = images.reshape(B, C, H//2, 2, W//2, 2)
    return separated.permute(0,2,4,1,3,5).contiguous().reshape(B,4,4)

class TinyViTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_projection = nn.Linear(4,8)
        self.class_token = nn.Parameter(torch.zeros(1,1,8))
        self.positions = nn.Parameter(torch.zeros(1,5,8))
        self.attention = nn.MultiheadAttention(8,1,dropout=0,batch_first=True)
        self.norm = nn.LayerNorm(8,eps=1e-5)
        self.head = nn.Linear(8,2)

    def forward(self, images, return_aux=False):
        if images.ndim != 4 or images.shape[1:] != (1,4,4) or images.dtype != torch.float32:
            raise ValueError("expected CPU float32 images (B,1,4,4)")
        patches = patchify(images)
        projected = self.patch_projection(patches)
        expanded_class = self.class_token.expand(images.shape[0],-1,-1)
        positioned = torch.cat((expanded_class, projected),dim=1) + self.positions
        attended, _ = self.attention(positioned, positioned, positioned, need_weights=False)
        normalized = self.norm(positioned + attended)
        class_output = normalized[:,0:1,:]
        logits = self.head(class_output[:,0,:])
        if return_aux:
            return logits, {"patches":patches, "positioned":positioned, "class_output":class_output}
        return logits

def compute_loss(model, features, targets):
    validate_actual(features, targets)
    logits = model(features["image"])
    if logits.shape != (features["image"].shape[0],2):
        raise ValueError("model must return (B,2) logits")
    return F.cross_entropy(logits, targets["label"], reduction="mean")

def train_vit_classifier(model, batch, optimizer):
    features, targets = batch
    role, ids = validate_actual(features, targets)
    if role != "train" or ids != TRAIN_IDS:
        raise ValueError("optimizer batches must contain only the immutable train split")
    trace = []
    model.train()
    for update in range(1,13):
        optimizer.zero_grad(set_to_none=True)
        loss = compute_loss(model, features, targets)
        loss.backward()
        optimizer.step()
        trace.append({"update":update, "mean_ce":float(loss.detach())})
    return tuple(trace)

train_batch = build_train_batch(TRAIN_IDS)
heldout_batch = build_heldout_batch(HELDOUT_IDS)
seed_everything()
model = TinyViTClassifier()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=.05, betas=(.9,.999), eps=1e-8, weight_decay=0,
    amsgrad=False, foreach=False, fused=False,
)
model.eval()
with torch.no_grad():
    heldout_loss_before = float(compute_loss(model, *heldout_batch))
training_trace = train_vit_classifier(model, train_batch, optimizer)
model.eval()
with torch.no_grad():
    heldout_loss_after = float(compute_loss(model, *heldout_batch))
    heldout_logits, heldout_aux = model(heldout_batch[0]["image"], return_aux=True)
    heldout_correct = int((heldout_logits.argmax(1) == heldout_batch[1]["label"]).sum())
certificate = {
    "updates": len(training_trace),
    "heldout_loss_before": heldout_loss_before,
    "heldout_loss_after": heldout_loss_after,
    "heldout_improvement": heldout_loss_before-heldout_loss_after,
    "heldout_correct": heldout_correct,
    "class_output_shape": tuple(heldout_aux["class_output"].shape),
}


### Answer check

In [ ]:
assert len(training_trace)==12
assert tuple(row["update"] for row in training_trace)==tuple(range(1,13))
assert certificate["class_output_shape"]==(2,1,8)
assert certificate["heldout_improvement"] >= .02
assert certificate["heldout_correct"] >= 2
assert set(TRAIN_IDS).isdisjoint(HELDOUT_IDS)
assert vision_fixture.validate_actual_batch('vit', _to_numpy(train_batch[0]), _to_numpy(train_batch[1])) == ('train', TRAIN_IDS)
assert vision_fixture.validate_actual_batch('vit', _to_numpy(heldout_batch[0]), _to_numpy(heldout_batch[1])) == ('heldout', HELDOUT_IDS)
bad_sequences=(TRAIN_IDS[::-1], TRAIN_IDS[:-1], (TRAIN_IDS[0],TRAIN_IDS[0],*TRAIN_IDS[2:]), (*TRAIN_IDS[:-1],'unknown-id'), HELDOUT_IDS)
for bad in bad_sequences:
    try:
        build_train_batch(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("train wrapper accepted an invalid ID sequence")
for bad in (HELDOUT_IDS[::-1], HELDOUT_IDS[:1], (HELDOUT_IDS[0],HELDOUT_IDS[0]), ('unknown-id',HELDOUT_IDS[1]), TRAIN_IDS):
    try:
        build_heldout_batch(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("held-out wrapper accepted an invalid ID sequence")
repeated_loss = compute_loss(model, *train_batch)
assert torch.isfinite(repeated_loss)
